In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 8 — PCA Dimensionality Reduction & Visualization

**Project:** Wholesale Customer Segmentation

Phase 8 - Dimensionality Reduction & Visualization
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 22. PCA dimensionality reduction (for visualization)
 23. Visualize clusters

Depends on: phase2_eda.py (SPEND_COLS, FIG_DIR), phase4_transform_scale.py
            output (scaled_features.csv), phase6_train_validate.py output
            (labeled_customers.csv)
Outputs: PNG figures saved to ./figures/, printed PCA diagnostics

Assumption (explicitly stated, carried from Phase 6/7):
 PCA and cluster visualization use the PRIMARY K=2 solution (stable, per
 Phase 6's ARI stability check), not the unstable K=3 alternative.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



# Recreate constants locally. Phase 6 supplies labels and Phase 4 supplies scaled features.
SPEND_COLS = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")

RANDOM_STATE = 42
N_INIT = 20
PRIMARY_CLUSTER_COL = "cluster_k2"
CHANNEL_MAP = {1: "Horeca (1)", 2: "Retail (2)"}

## SECTION 1: Load Scaled Features & Cluster Labels

In [ ]:
# SECTION 1: Load Scaled Features & Cluster Labels
# ===========================================================================
def load_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the scaled feature matrix (PCA input) and the cluster-labeled
    raw data (for coloring/overlay), and confirm they align row-for-row."""
    print("=" * 70)
    print("SECTION 1: LOAD SCALED FEATURES & CLUSTER LABELS")
    print("=" * 70)

    scaled_df = pd.read_csv("scaled_features.csv")
    labeled_df = pd.read_csv("labeled_customers.csv")

    assert len(scaled_df) == len(labeled_df), (
        "Row count mismatch between scaled_features.csv and labeled_customers.csv."
    )
    print(f"Scaled feature matrix shape: {scaled_df.shape}")
    print(f"Labeled data shape: {labeled_df.shape}")
    print(f"Using primary cluster column: '{PRIMARY_CLUSTER_COL}' (K=2, stable per Phase 6)")
    return scaled_df, labeled_df

## SECTION 2: Apply PCA — 2 Components (Step 22)

In [ ]:
# SECTION 2: Apply PCA — 2 Components (Step 22)
# ===========================================================================
def apply_pca(scaled_df: pd.DataFrame) -> tuple[np.ndarray, PCA]:
    """Reduce the 6 scaled spend features to 2 principal components for
    2D visualization. Not used for clustering itself (Phase 6 clustered on
    the full 6-feature space) — PCA here serves the visualization need
    flagged in Phase 3's multicollinearity decision (Option A)."""
    print("\n" + "=" * 70)
    print("SECTION 2: APPLY PCA (2 COMPONENTS)")
    print("=" * 70)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    pcs = pca.fit_transform(scaled_df[SPEND_COLS])

    explained_var = pca.explained_variance_ratio_
    print(f"PC1 explains {explained_var[0]*100:.1f}% of variance")
    print(f"PC2 explains {explained_var[1]*100:.1f}% of variance")
    print(f"Combined (PC1+PC2): {explained_var.sum()*100:.1f}% of total variance")

    loadings = pd.DataFrame(
        pca.components_.T, columns=["PC1", "PC2"], index=SPEND_COLS
    )
    print("\nPCA component loadings (feature contribution to each PC):")
    print(loadings.round(3).to_string())

    top_pc1 = loadings["PC1"].abs().idxmax()
    top_pc2 = loadings["PC2"].abs().idxmax()
    print(f"\n[FINDING] PC1 is most driven by '{top_pc1}' (consistent with Phase 7's "
          f"ANOVA finding that Grocery/Detergents_Paper/Milk dominate cluster "
          "separation). PC2 is most driven by "
          f"'{top_pc2}'.")

    return pcs, pca

## SECTION 3: PCA Loadings Visualization (supports Step 22 interpretation)

In [ ]:
# SECTION 3: PCA Loadings Visualization (supports Step 22 interpretation)
# ===========================================================================
def plot_pca_loadings(pca: PCA) -> None:
    """Bar chart of each feature's loading on PC1 and PC2, to make the
    PCA finding in Section 2 visually verifiable rather than table-only."""
    print("\n" + "=" * 70)
    print("SECTION 3: PCA LOADINGS VISUALIZATION")
    print("=" * 70)

    loadings = pd.DataFrame(pca.components_.T, columns=["PC1", "PC2"], index=SPEND_COLS)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    loadings["PC1"].sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Feature Loadings on PC1")
    axes[0].set_xlabel("Loading")

    loadings["PC2"].sort_values().plot(kind="barh", ax=axes[1], color="darkorange")
    axes[1].set_title("Feature Loadings on PC2")
    axes[1].set_xlabel("Loading")

    fig.suptitle("PCA Component Loadings by Feature", fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/20_pca_loadings.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/20_pca_loadings.png")

## SECTION 4: PCA Scatter Plot Colored by Cluster, Variance Annotated (Step 23)

In [ ]:
# SECTION 4: PCA Scatter Plot Colored by Cluster, Variance Annotated (Step 23)
# ===========================================================================
def plot_pca_cluster_scatter(pcs: np.ndarray, labeled_df: pd.DataFrame, pca: PCA) -> None:
    """Primary visualization: PC1 vs. PC2 scatter, colored by cluster label,
    with axis labels annotating the variance explained by each component."""
    print("\n" + "=" * 70)
    print("SECTION 4: PCA SCATTER PLOT COLORED BY CLUSTER")
    print("=" * 70)

    explained_var = pca.explained_variance_ratio_
    plot_df = pd.DataFrame({
        "PC1": pcs[:, 0],
        "PC2": pcs[:, 1],
        "Cluster": labeled_df[PRIMARY_CLUSTER_COL].astype(str),
    })

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="Cluster",
                     palette="Set2", s=60, alpha=0.75, ax=ax)
    ax.set_xlabel(f"PC1 ({explained_var[0]*100:.1f}% variance explained)")
    ax.set_ylabel(f"PC2 ({explained_var[1]*100:.1f}% variance explained)")
    ax.set_title(f"K-Means Clusters (K=2) Visualized in PCA Space\n"
                 f"(PC1+PC2 capture {explained_var.sum()*100:.1f}% of total variance)")
    ax.legend(title="Cluster")
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/21_pca_cluster_scatter.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/21_pca_cluster_scatter.png")
    print(f"[NOTE] Only {explained_var.sum()*100:.1f}% of total variance is captured "
          "in this 2D view — cluster separation may look less clean here than in "
          "the full 6-feature space K-Means actually used (Phase 6). This plot is "
          "for visualization only; it is not the basis the clustering itself relied on.")

## SECTION 5: PCA Scatter Overlaid with Channel Markers (Step 23, optional)

In [ ]:
# SECTION 5: PCA Scatter Overlaid with Channel Markers (Step 23, optional)
# ===========================================================================
def plot_pca_channel_overlay(pcs: np.ndarray, labeled_df: pd.DataFrame, pca: PCA) -> None:
    """Overlay Channel as marker shape on the same PCA scatter (color = cluster,
    marker shape = Channel) to visually compare cluster boundaries against the
    known Horeca/Retail split, following up on Phase 7's cross-tab finding."""
    print("\n" + "=" * 70)
    print("SECTION 5: PCA SCATTER OVERLAID WITH CHANNEL MARKERS")
    print("=" * 70)

    explained_var = pca.explained_variance_ratio_
    plot_df = pd.DataFrame({
        "PC1": pcs[:, 0],
        "PC2": pcs[:, 1],
        "Cluster": labeled_df[PRIMARY_CLUSTER_COL].astype(str),
        "Channel": labeled_df["Channel"].map(CHANNEL_MAP),
    })

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="Cluster", style="Channel",
                     palette="Set2", s=70, alpha=0.75, ax=ax)
    ax.set_xlabel(f"PC1 ({explained_var[0]*100:.1f}% variance explained)")
    ax.set_ylabel(f"PC2 ({explained_var[1]*100:.1f}% variance explained)")
    ax.set_title("Clusters (color) vs. Channel (marker shape) in PCA Space")
    ax.legend(title="Cluster / Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/22_pca_cluster_channel_overlay.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/22_pca_cluster_channel_overlay.png")
    print("[NOTE] This illustrates Phase 7's post-hoc cross-tab relationship "
          "(Cluster 1 = 96.8% Horeca, Cluster 0 = 71.7% Retail) — expect most "
          "color/shape combinations to align, with some mixed cases visible "
          "as the minority channel within each cluster's color region.")

## SECTION 6: Supporting Figure — Elbow Method (Step 23, "supporting figures")

In [ ]:
# SECTION 6: Supporting Figure — Elbow Method (Step 23, "supporting figures")
# ===========================================================================
def plot_supporting_elbow(scaled_df: pd.DataFrame) -> None:
    """Regenerate the elbow plot (originally computed in Phase 5) here so
    this phase's deliverable is self-contained, as the plan calls for the
    elbow chart to be included as a supporting figure alongside the PCA
    visualizations."""
    print("\n" + "=" * 70)
    print("SECTION 6: SUPPORTING FIGURE — ELBOW METHOD (K=2-10)")
    print("=" * 70)

    X = scaled_df[SPEND_COLS].values
    inertias = []
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        km.fit(X)
        inertias.append(km.inertia_)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(2, 11), inertias, marker="o", color="steelblue")
    ax.axvline(2, color="red", linestyle="--", alpha=0.6, label="Chosen K=2")
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
    ax.set_title("Supporting Figure: Elbow Method (originally computed in Phase 5)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/23_supporting_elbow.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/23_supporting_elbow.png "
          "(reproduced from Phase 5 for self-contained reference)")

## SECTION 7: Supporting Figure — Silhouette Plot for Final K (Step 23)

In [ ]:
# SECTION 7: Supporting Figure — Silhouette Plot for Final K (Step 23)
# ===========================================================================
def plot_supporting_silhouette(scaled_df: pd.DataFrame) -> None:
    """Regenerate the silhouette-vs-K plot (originally computed in Phase 5),
    marking the chosen K=2, as a supporting figure for this phase's
    visualization deliverable."""
    print("\n" + "=" * 70)
    print("SECTION 7: SUPPORTING FIGURE — SILHOUETTE SCORE BY K")
    print("=" * 70)

    X = scaled_df[SPEND_COLS].values
    sil_scores = []
    for k in range(2, 11):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = km.fit_predict(X)
        sil_scores.append(silhouette_score(X, labels))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(2, 11), sil_scores, marker="o", color="darkorange")
    ax.axvline(2, color="red", linestyle="--", alpha=0.6, label="Chosen K=2")
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel("Average Silhouette Score")
    ax.set_title("Supporting Figure: Silhouette Score by K (originally computed in Phase 5)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/24_supporting_silhouette.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/24_supporting_silhouette.png "
          f"(K=2 silhouette={sil_scores[0]:.4f}, reproduced from Phase 5 for reference)")

## MAIN — run Phase 8 end to end

In [ ]:
# MAIN — run Phase 8 end to end
# ===========================================================================
if __name__ == "__main__":
    scaled_df, labeled_df = load_data()
    pcs, pca = apply_pca(scaled_df)
    plot_pca_loadings(pca)
    plot_pca_cluster_scatter(pcs, labeled_df, pca)
    plot_pca_channel_overlay(pcs, labeled_df, pca)
    plot_supporting_elbow(scaled_df)
    plot_supporting_silhouette(scaled_df)

    print("\n" + "=" * 70)
    print("PHASE 8 COMPLETE")
    print("=" * 70)
    print(f"[OK] PCA applied (2 components), variance explained annotated.")
    print(f"[OK] Cluster scatter plot created (colored by K=2 cluster label).")
    print(f"[OK] Channel overlay scatter plot created (color=cluster, shape=Channel).")
    print(f"[OK] Supporting elbow and silhouette figures regenerated for this phase.")
    print("[OK] Ready for Phase 9 (Interpretation, Business Implications & Limitations).")

### Phase 8 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.